# Causal Abstraction

Tests alignment between model computations and hypothesized causal models via interchange interventions and abstraction quality metrics.

**References:**
- Geiger et al. (2021) "Causal Abstractions of Neural Networks"
- Geiger et al. (2023) "Finding Alignments Between Interpretable Causal Variables and Distributed Neural Representations"

## Why This Matters

Causal abstraction tests whether a high-level algorithm (like 'compare two numbers') is faithfully implemented by a neural circuit. Interchange interventions and distributed alignment search (DAS) provide rigorous methods for mapping between algorithmic descriptions and neural implementations.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.causal_abstraction import (
    interchange_intervention,
    causal_abstraction_test,
    distributed_alignment_search,
    multi_variable_alignment,
    abstraction_quality_score,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens_a = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
tokens_b = jnp.array([1, 6, 11, 16, 21, 26, 31, 36])
def metric(logits): return float(logits[-1, 0])
print(f"Model: {cfg.n_layers} layers")

In [ ]:
# 1. Interchange intervention
ii = interchange_intervention(model, tokens_a, tokens_b, "blocks.1.hook_resid_post")
print(f"KL divergence: {ii['kl_divergence']:.6f}")
print(f"Logit diff: {ii['logit_diff']:.6f}")

In [ ]:
# 2. Causal abstraction test
pairs = [(tokens_a, tokens_b, 0.1), (tokens_b, tokens_a, -0.1)]
cat = causal_abstraction_test(model, pairs, "blocks.1.hook_resid_post", metric)
print(f"Mean alignment: {cat['mean_alignment']:.4f}")
print(f"Alignment rate: {cat['alignment_rate']:.1%}")
print(f"Per-pair scores: {cat['alignment_scores'].round(4)}")

In [ ]:
# 3. Distributed alignment search
das = distributed_alignment_search(model, tokens_a, tokens_b, metric, layer=1)
print(f"Full patch effect: {das['full_patch_effect']:.6f}")
print(f"Dims for half effect: {das['dims_for_half_effect']}")
top_dims = np.argsort(np.abs(das['direction_effects']))[-5:][::-1]
print(f"Top dimensions: {top_dims.tolist()}")
print(f"Effects: {das['direction_effects'][top_dims].round(6)}")

In [ ]:
# 4. Multi-variable alignment
hooks = [f"blocks.{l}.hook_attn_out" for l in range(cfg.n_layers)]
mva = multi_variable_alignment(model, [tokens_a, tokens_b], hooks, metric)
print(f"Rankings: {mva['hook_rankings']}")
print(f"Complementarity: {mva['complementarity']:.4f}")
for name, eff in sorted(mva['hook_effects'].items(), key=lambda x: -x[1]):
    print(f"  {name}: {eff:.6f}")

In [ ]:
# 5. Abstraction quality score
for l in range(cfg.n_layers):
    hook = f"blocks.{l}.hook_resid_post"
    aq = abstraction_quality_score(model, tokens_a, tokens_b, hook, metric)
    print(f"Layer {l}: faith={aq['faithfulness']:.3f}, spec={aq['specificity']:.3f}, quality={aq['quality_score']:.3f}")